# 02 Scan Metadata to Staging

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime
import uuid, re
CATALOG_SCHEMA="governance"
sync_run_id=datetime.now().strftime("%Y%m%d_%H%M%S")+"_"+str(uuid.uuid4())[:8]
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_SCHEMA}")
print(f"sync_run_id={sync_run_id}")

def q(n):
    if n is None or str(n).strip()=="": return None
    return f"`{str(n).strip().replace('`','``')}`"
def ns(w,l,s=None):
    parts=[w,l]+([] if s is None or str(s).strip()=="" else [s])
    return ".".join(q(x) for x in parts)
def full(w,l,s,t): return f"{ns(w,l,s)}.{q(t)}"

cfg=(spark.table(f"{CATALOG_SCHEMA}.cfg_metadata_source")
     .where(F.col("is_active")==True)
     .withColumn("include_table_pattern",F.coalesce(F.col("include_table_pattern"),F.lit(".*")))
     .withColumn("exclude_table_pattern",F.coalesce(F.col("exclude_table_pattern"),F.lit(""))))

table_rows=[]; column_rows=[]
for src in cfg.collect():
    source_id,layer,w,l,s,domain,owner=src['source_id'],src['layer'],src['workspace_name'],src['lakehouse_name'],src['schema_name'],src['domain'],src['owner_team']
    include_pat=src['include_table_pattern'] or ".*"; exclude_pat=src['exclude_table_pattern'] or ""; scan_row_count=bool(src['scan_row_count'])
    namespace=ns(w,l,s)
    print('Scanning',source_id,namespace)
    try:
        names=[r['tableName'] for r in spark.sql(f"SHOW TABLES IN {namespace}").where(F.col('isTemporary')==False).select('tableName').distinct().collect()]
    except Exception as e:
        table_rows.append({'sync_run_id':sync_run_id,'source_id':source_id,'layer':layer,'workspace_name':w,'lakehouse_name':l,'schema_name':s,'table_name':None,'domain':domain,'owner_team':owner,'row_count':None,'column_count':None,'scan_status':'Failed','error_message':f'SHOW TABLES failed: {str(e)[:3500]}'})
        continue
    for t in names:
        if not re.match(include_pat,t): continue
        if exclude_pat and re.match(exclude_pat,t): continue
        try:
            df=spark.table(full(w,l,s,t)); fields=df.schema.fields; row_count=df.count() if scan_row_count else None
            table_rows.append({'sync_run_id':sync_run_id,'source_id':source_id,'layer':layer,'workspace_name':w,'lakehouse_name':l,'schema_name':s,'table_name':t,'domain':domain,'owner_team':owner,'row_count':row_count,'column_count':len(fields),'scan_status':'Success','error_message':None})
            for i,f in enumerate(fields,1):
                column_rows.append({'sync_run_id':sync_run_id,'source_id':source_id,'layer':layer,'workspace_name':w,'lakehouse_name':l,'schema_name':s,'table_name':t,'column_name':f.name,'ordinal_position':i,'data_type':f.dataType.simpleString(),'is_nullable':f.nullable,'scan_status':'Success','error_message':None})
        except Exception as e:
            table_rows.append({'sync_run_id':sync_run_id,'source_id':source_id,'layer':layer,'workspace_name':w,'lakehouse_name':l,'schema_name':s,'table_name':t,'domain':domain,'owner_team':owner,'row_count':None,'column_count':None,'scan_status':'Failed','error_message':str(e)[:3500]})

table_schema=StructType([StructField('sync_run_id',StringType(),False),StructField('source_id',StringType(),True),StructField('layer',StringType(),True),StructField('workspace_name',StringType(),True),StructField('lakehouse_name',StringType(),True),StructField('schema_name',StringType(),True),StructField('table_name',StringType(),True),StructField('domain',StringType(),True),StructField('owner_team',StringType(),True),StructField('row_count',LongType(),True),StructField('column_count',IntegerType(),True),StructField('scan_status',StringType(),True),StructField('error_message',StringType(),True)])
column_schema=StructType([StructField('sync_run_id',StringType(),False),StructField('source_id',StringType(),True),StructField('layer',StringType(),True),StructField('workspace_name',StringType(),True),StructField('lakehouse_name',StringType(),True),StructField('schema_name',StringType(),True),StructField('table_name',StringType(),True),StructField('column_name',StringType(),True),StructField('ordinal_position',IntegerType(),True),StructField('data_type',StringType(),True),StructField('is_nullable',BooleanType(),True),StructField('scan_status',StringType(),True),StructField('error_message',StringType(),True)])

spark.createDataFrame(table_rows,table_schema).withColumn('scanned_at',F.current_timestamp()).withColumn('sync_date',F.current_date()).write.format('delta').mode('append').partitionBy('sync_date').saveAsTable(f'{CATALOG_SCHEMA}.stg_table_metadata')
spark.createDataFrame(column_rows,column_schema).withColumn('scanned_at',F.current_timestamp()).withColumn('sync_date',F.current_date()).write.format('delta').mode('append').partitionBy('sync_date').saveAsTable(f'{CATALOG_SCHEMA}.stg_column_metadata')
print('Staging written')
